In [7]:
import os
import time
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, roc_auc_score

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("PyTorch version:", torch.__version__)


PyTorch version: 2.9.1+cpu


In [8]:
# Limpieza opcional de cache de pesos si el hash falla
CLEAR_TORCH_CACHE = False  # Cambia a True si tienes error de hash
if CLEAR_TORCH_CACHE:
    cache_path = Path.home() / ".cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth"
    if cache_path.exists():
        print("Removing cached weights:", cache_path)
        cache_path.unlink()
    else:
        print("Cache file not found:", cache_path)


In [9]:
# Rutas de datos (splits generados previamente)
BASE_DIR = Path("../data/processed/classification_split")
TRAIN_DIR = BASE_DIR / "train"
VAL_DIR = BASE_DIR / "val"
TEST_DIR = BASE_DIR / "test"

IMG_SIZE = 224
BATCH_SIZE = 32

print("Train:", TRAIN_DIR)
print("Val:", VAL_DIR)
print("Test:", TEST_DIR)

# Validacion de rutas
for p in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    if not p.exists():
        raise FileNotFoundError(f"No existe: {p}")


Train: ..\data\processed\classification_split\train
Val: ..\data\processed\classification_split\val
Test: ..\data\processed\classification_split\test


In [10]:
# Transforms (equivalentes al aumento en TensorFlow)
train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0, translate=(0.08, 0.08)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_ds = datasets.ImageFolder(TRAIN_DIR, transform=train_tfms)
val_ds = datasets.ImageFolder(VAL_DIR, transform=eval_tfms)
test_ds = datasets.ImageFolder(TEST_DIR, transform=eval_tfms)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=torch.cuda.is_available())
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())
test_dl = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())

print("Classes:", train_ds.classes)


Classes: ['mosaico_dorado', 'sano']


In [11]:
# Modelo MobileNetV2 + head binario con Swish (SiLU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

try:
    base = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
except Exception as e:
    print("Warning: failed to load ImageNet weights, using random init.", e)
    base = models.mobilenet_v2(weights=None)

for p in base.features.parameters():
    p.requires_grad = False

base.classifier = nn.Sequential(
    nn.Dropout(0.2),
    nn.Linear(base.last_channel, 128),
    nn.SiLU(),  # Swish
    nn.Dropout(0.2),
    nn.Linear(128, 1),
)

model = base.to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=1e-3)


Device: cpu


In [12]:
# Entrenamiento
EPOCHS = 30
best_acc = 0.0
best_state = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_correct = 0
    train_total = 0
    for xb, yb in train_dl:
        xb = xb.to(device)
        yb = yb.to(device).float()
        optimizer.zero_grad()
        out = model(xb).squeeze(1)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        preds = (torch.sigmoid(out) > 0.5).long()
        train_correct += (preds == yb.long()).sum().item()
        train_total += yb.size(0)
    train_acc = train_correct / max(1, train_total)

    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for xb, yb in val_dl:
            xb = xb.to(device)
            yb = yb.to(device).float()
            out = model(xb).squeeze(1)
            preds = (torch.sigmoid(out) > 0.5).long()
            val_correct += (preds == yb.long()).sum().item()
            val_total += yb.size(0)
    val_acc = val_correct / max(1, val_total)
    if val_acc > best_acc:
        best_acc = val_acc
        best_state = {k: v.cpu() for k, v in model.state_dict().items()}
    print(f"Epoch {epoch:02d} | train_acc={train_acc:.4f} | val_acc={val_acc:.4f}")

if best_state is not None:
    model.load_state_dict(best_state)


Epoch 01 | train_acc=0.9442 | val_acc=0.9602
Epoch 02 | train_acc=0.9881 | val_acc=0.9906
Epoch 03 | train_acc=0.9897 | val_acc=0.9883
Epoch 04 | train_acc=0.9943 | val_acc=0.9883
Epoch 05 | train_acc=0.9912 | val_acc=0.9930
Epoch 06 | train_acc=0.9928 | val_acc=0.9930
Epoch 07 | train_acc=0.9881 | val_acc=0.9883
Epoch 08 | train_acc=0.9850 | val_acc=0.9930
Epoch 09 | train_acc=0.9923 | val_acc=0.9930
Epoch 10 | train_acc=0.9943 | val_acc=0.9883
Epoch 11 | train_acc=0.9897 | val_acc=0.9883
Epoch 12 | train_acc=0.9907 | val_acc=0.9859
Epoch 13 | train_acc=0.9954 | val_acc=0.9859
Epoch 14 | train_acc=0.9763 | val_acc=0.9836
Epoch 15 | train_acc=0.9897 | val_acc=0.9930
Epoch 16 | train_acc=0.9876 | val_acc=0.9883
Epoch 17 | train_acc=0.9948 | val_acc=0.9930
Epoch 18 | train_acc=0.9948 | val_acc=0.9789
Epoch 19 | train_acc=0.9892 | val_acc=0.9930
Epoch 20 | train_acc=0.9964 | val_acc=0.9906
Epoch 21 | train_acc=0.9979 | val_acc=0.9930
Epoch 22 | train_acc=0.9912 | val_acc=0.9906
Epoch 23 |

In [13]:
# Evaluacion en test
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for xb, yb in test_dl:
        xb = xb.to(device)
        yb = yb.to(device).float()
        out = model(xb).squeeze(1)
        preds = (torch.sigmoid(out) > 0.5).long()
        correct += (preds == yb.long()).sum().item()
        total += yb.size(0)
test_acc = correct / max(1, total)
print("Test Accuracy:", test_acc)


Test Accuracy: 0.9950617283950617


In [14]:
# Metricas adicionales en test
model.eval()
all_probs = []
all_preds = []
all_labels = []
with torch.no_grad():
    for xb, yb in test_dl:
        xb = xb.to(device)
        yb = yb.to(device).float()
        out = model(xb).squeeze(1)
        probs = torch.sigmoid(out)
        preds = (probs > 0.5).long()
        all_probs.append(probs.cpu().numpy())
        all_preds.append(preds.cpu().numpy())
        all_labels.append(yb.long().cpu().numpy())
y_true = np.concatenate(all_labels)
y_pred = np.concatenate(all_preds)
y_prob = np.concatenate(all_probs)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="binary", zero_division=0
)
auc = roc_auc_score(y_true, y_prob)
cm = confusion_matrix(y_true, y_pred)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("AUC:", auc)
print("Confusion Matrix:\n", cm)


Precision: 0.991701244813278
Recall: 1.0
F1: 0.9958333333333333
AUC: 0.9999243837273781
Confusion Matrix:
 [[164   2]
 [  0 239]]


In [15]:
# Guardar modelos en la carpeta models de la raiz
out_dir = Path("../models/classification")
out_dir.mkdir(parents=True, exist_ok=True)
state_path = out_dir / "pytorch_mobilenetv2.pth"
full_path = out_dir / "pytorch_mobilenetv2_full.pt"
torch.save(model.state_dict(), state_path)
torch.save(model, full_path)
print("Saved:", state_path)
print("Saved:", full_path)


Saved: ..\models\classification\pytorch_mobilenetv2.pth
Saved: ..\models\classification\pytorch_mobilenetv2_full.pt
